# OSRM vs Abs_PM vs 经纬度距离 验证实验

## 目标
1. 对同 (Fwy, Dir) 的站点对，计算三种距离
2. 验证 OSRM 双向距离与 Abs_PM 方向的一致性
3. **在地图上可视化方向判断错误的案例，分析原因**

## 筛选条件
- 可指定特定高速公路，或全量计算
- 同一高速 (Fwy) 同一方向 (Dir)
- 经纬度距离: 0.1 mi < dist_geo < 4 mi

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import os
import re
import glob
import json
from datetime import datetime
from math import radians, sin, cos, sqrt, atan2
from tqdm import tqdm
import matplotlib.pyplot as plt
import folium
from folium import plugins
import warnings
warnings.filterwarnings('ignore')

# ============== 配置 ==============
META_DIR = "../d03_meta"
OUTPUT_DIR = "../output/osrm_validation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# OSRM 服务器
OSRM_SERVER = "http://router.project-osrm.org"  # 公共服务器（慢）
# OSRM_SERVER = "http://localhost:5000"  # 本地服务器（快）

# 本地服务器几乎不需要延迟
RATE_LIMIT_DELAY = 0.01  # 秒

# 距离筛选条件（英里）
MIN_GEO_DISTANCE = 0.1  # 最小距离，排除同位置站点
MAX_GEO_DISTANCE = 4.0  # 最大距离

# ============== 高速公路筛选 ==============
# 设置为 None 则全量计算所有高速
# 设置为具体值则只计算该高速，例如: "80", "50", "5"
TARGET_FWY = "80"  # 只计算 I-80

# 可选：进一步限定方向，设置为 None 则两个方向都算
# TARGET_DIR = None  # 两个方向都算
TARGET_DIR = "E"  # 或 "N", "S", "E", "W"

print("配置完成！")
print(f"OSRM 服务器: {OSRM_SERVER}")
print(f"距离范围: {MIN_GEO_DISTANCE} ~ {MAX_GEO_DISTANCE} mi")
print(f"目标高速: {TARGET_FWY if TARGET_FWY else '全部'}")
print(f"目标方向: {TARGET_DIR if TARGET_DIR else '全部'}")

配置完成！
OSRM 服务器: http://router.project-osrm.org
距离范围: 0.1 ~ 4.0 mi
目标高速: 80
目标方向: E


## 1. 加载元数据

In [2]:
# 查找最新的元数据文件
meta_files = glob.glob(os.path.join(META_DIR, "d03_text_meta_*.txt"))
if meta_files:
    meta_file = sorted(meta_files)[-1]
else:
    raise FileNotFoundError("未找到 D3 元数据文件")

print(f"使用元数据: {meta_file}")

META_COLUMNS = [
    'ID', 'Fwy', 'Dir', 'District', 'County', 'City',
    'State_PM', 'Abs_PM', 'Latitude', 'Longitude', 'Length',
    'Type', 'Lanes', 'Name', 'User_ID_1', 'User_ID_2',
    'User_ID_3', 'User_ID_4'
]

meta_df = pd.read_csv(
    meta_file, sep='\t', names=META_COLUMNS, header=0,
    dtype={'ID': str, 'Fwy': str}
)

print(f"总站点数: {len(meta_df)}")
print(f"\n各类型站点数:")
print(meta_df['Type'].value_counts())

# 显示可用的高速公路列表
print(f"\n可用高速公路:")
fwy_list = meta_df.groupby(['Fwy', 'Dir']).size().reset_index(name='Count')
print(fwy_list.to_string())

使用元数据: ../d03_meta/d03_text_meta_2025_12_30.txt
总站点数: 1903

各类型站点数:
Type
ML    884
OR    431
HV    281
FR    279
FF     28
Name: count, dtype: int64

可用高速公路:
    Fwy Dir  Count
0   113   N      9
1   113   S     10
2    12   E      9
3    12   W      7
4    16   W      1
5   160   N      5
6   160   S      5
7   162   E      3
8   162   W      3
9   193   E      2
10  193   W      1
11   20   E     27
12   20   W     22
13  244   E      2
14  244   W      3
15  267   E      2
16  267   W      2
17  275   W      2
18   28   E      3
19   28   W      3
20   45   N      3
21   45   S      3
22   49   N      6
23   49   S      6
24    5   N    144
25    5   S    141
26   50   E    243
27   50   W    223
28  505   N      1
29  505   S      1
30   51   N     49
31   51   S     41
32   65   N     29
33   65   S     36
34   70   E     13
35   70   W     11
36   80   E    245
37   80   W    235
38   89   N      9
39   89   S      9
40   99   N    159
41   99   S    175


In [3]:
# 坐标有效性检查
def is_valid_coord(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False
    if not (32 <= lat <= 42):
        return False
    if not (-124 <= lon <= -114):
        return False
    return True

meta_df['Valid_Coord'] = meta_df.apply(
    lambda r: is_valid_coord(r['Latitude'], r['Longitude']), axis=1
)

print(f"有效坐标站点: {meta_df['Valid_Coord'].sum()} / {len(meta_df)}")

meta_valid = meta_df[meta_df['Valid_Coord']].copy()

# 应用高速公路筛选
if TARGET_FWY:
    meta_valid = meta_valid[meta_valid['Fwy'] == TARGET_FWY]
    print(f"筛选高速 {TARGET_FWY} 后: {len(meta_valid)} 站点")

if TARGET_DIR:
    meta_valid = meta_valid[meta_valid['Dir'] == TARGET_DIR]
    print(f"筛选方向 {TARGET_DIR} 后: {len(meta_valid)} 站点")

print(f"\n最终站点数: {len(meta_valid)}")
print(f"\n筛选后各类型站点数:")
print(meta_valid['Type'].value_counts())

有效坐标站点: 1901 / 1903
筛选高速 80 后: 480 站点
筛选方向 E 后: 245 站点

最终站点数: 245

筛选后各类型站点数:
Type
ML    105
OR     51
HV     45
FR     42
FF      2
Name: count, dtype: int64


## 2. 距离计算函数

In [4]:
def calc_distance_pm(pm1, pm2):
    """Abs_PM 差值（有符号）"""
    return pm2 - pm1


def calc_distance_geo(lat1, lon1, lat2, lon2):
    """Haversine 直线距离（英里）"""
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c


def calc_distance_osrm_with_geometry(lat1, lon1, lat2, lon2, server=OSRM_SERVER, timeout=10):
    """
    OSRM 最短路距离（单向），返回路径几何
    """
    url = f"{server}/route/v1/driving/{lon1},{lat1};{lon2},{lat2}"
    params = {
        "overview": "full",
        "geometries": "geojson",
        "steps": "false"
    }
    
    try:
        response = requests.get(url, params=params, timeout=timeout)
        data = response.json()
        
        if data.get("code") != "Ok":
            return {
                'distance_mi': None,
                'duration_min': None,
                'geometry': None,
                'status': 'no_route',
                'error_msg': data.get("code", "Unknown")
            }
        
        route = data["routes"][0]
        return {
            'distance_mi': route["distance"] / 1609.34,
            'duration_min': route["duration"] / 60,
            'geometry': route["geometry"],
            'status': 'ok',
            'error_msg': None
        }
        
    except Exception as e:
        return {
            'distance_mi': None,
            'duration_min': None,
            'geometry': None,
            'status': 'error',
            'error_msg': str(e)
        }


# 测试 OSRM 连接
print("测试 OSRM 服务器连接...")
test_result = calc_distance_osrm_with_geometry(38.5, -121.5, 38.6, -121.4, OSRM_SERVER)
if test_result['status'] == 'ok':
    print(f"✓ OSRM 服务器正常，测试距离: {test_result['distance_mi']:.2f} mi")
else:
    print(f"✗ OSRM 连接失败: {test_result['error_msg']}")
    print("请检查 OSRM 服务器是否启动")

测试 OSRM 服务器连接...
✓ OSRM 服务器正常，测试距离: 12.80 mi


## 3. 构建符合条件的站点对

In [5]:
def build_all_pairs(meta_df, min_dist=0.1, max_dist=4.0):
    """
    构建所有符合条件的站点对
    条件: 同 (Fwy, Dir)，经纬度距离在 [min_dist, max_dist] 范围内
    """
    all_pairs = []
    
    for (fwy, direction), group in tqdm(meta_df.groupby(['Fwy', 'Dir']), desc="构建站点对"):
        sorted_group = group.sort_values('Abs_PM').reset_index(drop=True)
        n = len(sorted_group)
        
        if n < 2:
            continue
        
        # 遍历所有点对
        for i in range(n):
            for j in range(i + 1, n):
                node_i = sorted_group.iloc[i]
                node_j = sorted_group.iloc[j]
                
                # 计算经纬度距离
                dist_geo = calc_distance_geo(
                    node_i['Latitude'], node_i['Longitude'],
                    node_j['Latitude'], node_j['Longitude']
                )
                
                # 筛选距离范围
                if dist_geo < min_dist or dist_geo > max_dist:
                    continue
                
                all_pairs.append({
                    'ID1': node_i['ID'],
                    'ID2': node_j['ID'],
                    'Fwy': fwy,
                    'Dir': direction,
                    'Type1': node_i['Type'],
                    'Type2': node_j['Type'],
                    'PM1': node_i['Abs_PM'],
                    'PM2': node_j['Abs_PM'],
                    'Lat1': node_i['Latitude'],
                    'Lon1': node_i['Longitude'],
                    'Lat2': node_j['Latitude'],
                    'Lon2': node_j['Longitude'],
                    'Name1': node_i['Name'],
                    'Name2': node_j['Name'],
                    'Lanes1': node_i['Lanes'],
                    'Lanes2': node_j['Lanes'],
                    'Dist_Geo': dist_geo,
                })
    
    return pd.DataFrame(all_pairs)


# 构建符合条件的站点对
pairs_df = build_all_pairs(meta_valid, MIN_GEO_DISTANCE, MAX_GEO_DISTANCE)

# 计算 PM 距离
pairs_df['Dist_PM'] = pairs_df['PM2'] - pairs_df['PM1']

print(f"\n符合条件的站点对总数: {len(pairs_df)}")
print(f"覆盖 (Fwy, Dir) 数: {pairs_df.groupby(['Fwy', 'Dir']).ngroups}")
print(f"\n经纬度距离分布:")
print(pairs_df['Dist_Geo'].describe())
print(f"\n站点类型组合:")
print(pairs_df.groupby(['Type1', 'Type2']).size().sort_values(ascending=False).head(10))

构建站点对: 100%|██████████| 1/1 [00:03<00:00,  3.29s/it]


符合条件的站点对总数: 3814
覆盖 (Fwy, Dir) 数: 1

经纬度距离分布:
count    3814.000000
mean        2.134268
std         1.117006
min         0.100203
25%         1.233632
50%         2.122676
75%         3.126807
max         3.999977
Name: Dist_Geo, dtype: float64

站点类型组合:
Type1  Type2
ML     ML       601
       HV       409
HV     ML       395
       HV       359
OR     ML       274
ML     OR       256
FR     ML       224
ML     FR       223
OR     HV       178
HV     OR       163
dtype: int64


## 4. 批量 OSRM 双向查询

In [6]:
def batch_osrm_query_with_geometry(pairs_df, server=OSRM_SERVER, rate_limit=0.01):
    """
    批量 OSRM 双向查询，保存路径几何
    """
    results = []
    
    for idx, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="OSRM查询"):
        # 1 → 2
        r_1to2 = calc_distance_osrm_with_geometry(
            row['Lat1'], row['Lon1'],
            row['Lat2'], row['Lon2'],
            server
        )
        if rate_limit > 0:
            time.sleep(rate_limit)
        
        # 2 → 1
        r_2to1 = calc_distance_osrm_with_geometry(
            row['Lat2'], row['Lon2'],
            row['Lat1'], row['Lon1'],
            server
        )
        if rate_limit > 0:
            time.sleep(rate_limit)
        
        results.append({
            'idx': idx,
            'OSRM_1to2_mi': r_1to2['distance_mi'],
            'OSRM_2to1_mi': r_2to1['distance_mi'],
            'OSRM_1to2_min': r_1to2['duration_min'],
            'OSRM_2to1_min': r_2to1['duration_min'],
            'Geometry_1to2': json.dumps(r_1to2['geometry']) if r_1to2['geometry'] else None,
            'Geometry_2to1': json.dumps(r_2to1['geometry']) if r_2to1['geometry'] else None,
            'Status_1to2': r_1to2['status'],
            'Status_2to1': r_2to1['status'],
        })
    
    return pd.DataFrame(results).set_index('idx')


# 执行查询
print(f"开始 OSRM 查询，共 {len(pairs_df)} 对...")
est_time = len(pairs_df) * 2 * max(RATE_LIMIT_DELAY, 0.005) / 60
print(f"预计时间: {est_time:.1f} 分钟")
print("="*60)

osrm_results = batch_osrm_query_with_geometry(pairs_df, OSRM_SERVER, RATE_LIMIT_DELAY)

# 合并结果
pairs_df = pairs_df.join(osrm_results)

print(f"\n查询完成!")
print(f"成功率 (1→2): {(pairs_df['Status_1to2'] == 'ok').mean()*100:.1f}%")
print(f"成功率 (2→1): {(pairs_df['Status_2to1'] == 'ok').mean()*100:.1f}%")

开始 OSRM 查询，共 3814 对...
预计时间: 1.3 分钟


OSRM查询:   9%|▉         | 354/3814 [11:48<1:55:28,  2.00s/it]


KeyboardInterrupt: 

## 5. 方向判断与一致性分析

In [ ]:
# 只分析双向都成功的
valid_pairs = pairs_df[
    (pairs_df['Status_1to2'] == 'ok') & 
    (pairs_df['Status_2to1'] == 'ok')
].copy()

print(f"双向都成功的站点对: {len(valid_pairs)} / {len(pairs_df)}")

In [ ]:
def determine_direction_by_pm(row):
    """
    根据 Abs_PM 和 Dir 判断方向
    返回: '1to2' 或 '2to1'
    
    逻辑:
    - N/E 方向: PM 递增 = 车流方向，即 PM小→PM大
    - S/W 方向: PM 递减 = 车流方向，即 PM大→PM小
    
    由于我们构建点对时 node_i 的 PM < node_j 的 PM (按PM排序):
    - ID1 = PM 小的站点
    - ID2 = PM 大的站点
    """
    direction = row['Dir']
    
    if direction in ['N', 'E']:
        # N/E: 车流沿 PM 递增方向，即 ID1 → ID2
        return '1to2'
    else:
        # S/W: 车流沿 PM 递减方向，即 ID2 → ID1
        return '2to1'


def determine_direction_by_osrm(row, threshold_ratio=0.8):
    """
    根据 OSRM 双向距离判断方向
    距离短的是顺向
    """
    d_1to2 = row['OSRM_1to2_mi']
    d_2to1 = row['OSRM_2to1_mi']
    
    if pd.isna(d_1to2) or pd.isna(d_2to1):
        return 'unknown'
    
    if d_1to2 < d_2to1 * threshold_ratio:
        return '1to2'
    elif d_2to1 < d_1to2 * threshold_ratio:
        return '2to1'
    else:
        return 'similar'


# 计算方向判断
valid_pairs['Dir_by_PM'] = valid_pairs.apply(determine_direction_by_pm, axis=1)
valid_pairs['Dir_by_OSRM'] = valid_pairs.apply(determine_direction_by_osrm, axis=1)
valid_pairs['Direction_Match'] = valid_pairs['Dir_by_PM'] == valid_pairs['Dir_by_OSRM']

# 计算更多指标
valid_pairs['Abs_Dist_PM'] = valid_pairs['Dist_PM'].abs()
valid_pairs['OSRM_Forward_mi'] = valid_pairs[['OSRM_1to2_mi', 'OSRM_2to1_mi']].min(axis=1)
valid_pairs['OSRM_Backward_mi'] = valid_pairs[['OSRM_1to2_mi', 'OSRM_2to1_mi']].max(axis=1)
valid_pairs['OSRM_Ratio'] = valid_pairs['OSRM_Backward_mi'] / valid_pairs['OSRM_Forward_mi']

print("方向判断结果:")
print(f"\nPM 方向分布:")
print(valid_pairs['Dir_by_PM'].value_counts())
print(f"\nOSRM 方向分布:")
print(valid_pairs['Dir_by_OSRM'].value_counts())

In [ ]:
# 一致性统计
print("="*60)
print("方向判断一致性分析")
print("="*60)

# OSRM 有明确方向判断的
decisive = valid_pairs[valid_pairs['Dir_by_OSRM'].isin(['1to2', '2to1'])].copy()
print(f"\nOSRM 有明确方向判断: {len(decisive)} / {len(valid_pairs)} ({len(decisive)/len(valid_pairs)*100:.1f}%)")

if len(decisive) > 0:
    match_count = decisive['Direction_Match'].sum()
    match_rate = match_count / len(decisive) * 100
    print(f"方向一致: {match_count} ({match_rate:.1f}%)")
    print(f"方向不一致: {len(decisive) - match_count} ({100-match_rate:.1f}%)")

# OSRM 双向相近的
similar = valid_pairs[valid_pairs['Dir_by_OSRM'] == 'similar']
print(f"\nOSRM 双向距离相近: {len(similar)} ({len(similar)/len(valid_pairs)*100:.1f}%)")

In [ ]:
# 找出方向判断不一致的案例
mismatch = decisive[~decisive['Direction_Match']].copy()
print(f"\n方向不一致的案例数: {len(mismatch)}")

if len(mismatch) > 0:
    print("\n不一致案例详情:")
    display_cols = [
        'ID1', 'ID2', 'Fwy', 'Dir', 'Type1', 'Type2',
        'PM1', 'PM2', 'Dist_PM', 'Dist_Geo',
        'OSRM_1to2_mi', 'OSRM_2to1_mi', 'OSRM_Ratio',
        'Dir_by_PM', 'Dir_by_OSRM',
        'Name1', 'Name2'
    ]
    display(mismatch[display_cols])

## 6. 可视化方向判断错误的路径（OSM 底图）

In [ ]:
def create_error_visualization_map(mismatch_df, output_path, max_cases=50):
    """
    在地图上可视化方向判断错误的案例
    使用 OpenStreetMap 底图（与 OSRM 路径最匹配）
    
    - 蓝色路径: 1→2 方向的 OSRM 路径
    - 红色路径: 2→1 方向的 OSRM 路径
    - 绿色标记: 站点1 (PM 较小)
    - 红色标记: 站点2 (PM 较大)
    """
    if len(mismatch_df) == 0:
        print("没有错误案例需要可视化")
        return None
    
    # 限制数量
    if len(mismatch_df) > max_cases:
        print(f"错误案例过多，只显示前 {max_cases} 个")
        mismatch_df = mismatch_df.head(max_cases)
    
    # 计算地图中心
    center_lat = mismatch_df[['Lat1', 'Lat2']].values.mean()
    center_lon = mismatch_df[['Lon1', 'Lon2']].values.mean()
    
    # 创建地图 (OpenStreetMap 底图 - 与 OSRM 路径最匹配)
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=12,
        tiles='OpenStreetMap'
    )
    
    # 为每个错误案例添加标记和路径
    for idx, row in mismatch_df.iterrows():
        # 站点信息
        popup1 = f"""
        <b>站点1 (PM小)</b><br>
        ID: {row['ID1']}<br>
        Fwy: {row['Fwy']}-{row['Dir']}<br>
        Type: {row['Type1']}<br>
        Abs_PM: {row['PM1']:.3f}<br>
        Name: {row['Name1']}<br>
        Lanes: {row['Lanes1']}
        """
        
        popup2 = f"""
        <b>站点2 (PM大)</b><br>
        ID: {row['ID2']}<br>
        Fwy: {row['Fwy']}-{row['Dir']}<br>
        Type: {row['Type2']}<br>
        Abs_PM: {row['PM2']:.3f}<br>
        Name: {row['Name2']}<br>
        Lanes: {row['Lanes2']}
        """
        
        # 添加站点标记
        folium.Marker(
            [row['Lat1'], row['Lon1']],
            popup=folium.Popup(popup1, max_width=300),
            tooltip=f"{row['ID1']} ({row['Type1']}) PM={row['PM1']:.2f}",
            icon=folium.Icon(color='green', icon='info-sign')
        ).add_to(m)
        
        folium.Marker(
            [row['Lat2'], row['Lon2']],
            popup=folium.Popup(popup2, max_width=300),
            tooltip=f"{row['ID2']} ({row['Type2']}) PM={row['PM2']:.2f}",
            icon=folium.Icon(color='red', icon='info-sign')
        ).add_to(m)
        
        # 绘制 1→2 路径（蓝色）
        if row['Geometry_1to2']:
            geom_1to2 = json.loads(row['Geometry_1to2'])
            coords_1to2 = [[c[1], c[0]] for c in geom_1to2['coordinates']]
            folium.PolyLine(
                coords_1to2,
                color='blue',
                weight=4,
                opacity=0.8,
                popup=folium.Popup(
                    f"1→2: {row['OSRM_1to2_mi']:.2f} mi<br>"
                    f"PM判断: {row['Dir_by_PM']}<br>"
                    f"OSRM判断: {row['Dir_by_OSRM']}",
                    max_width=200
                ),
                tooltip=f"1→2: {row['OSRM_1to2_mi']:.2f} mi"
            ).add_to(m)
        
        # 绘制 2→1 路径（红色）
        if row['Geometry_2to1']:
            geom_2to1 = json.loads(row['Geometry_2to1'])
            coords_2to1 = [[c[1], c[0]] for c in geom_2to1['coordinates']]
            folium.PolyLine(
                coords_2to1,
                color='red',
                weight=4,
                opacity=0.6,
                popup=folium.Popup(
                    f"2→1: {row['OSRM_2to1_mi']:.2f} mi<br>"
                    f"PM判断: {row['Dir_by_PM']}<br>"
                    f"OSRM判断: {row['Dir_by_OSRM']}",
                    max_width=200
                ),
                tooltip=f"2→1: {row['OSRM_2to1_mi']:.2f} mi"
            ).add_to(m)
    
    # 添加图例
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
                background-color: white; padding: 10px; border: 2px solid gray;
                border-radius: 5px; font-size: 14px;">
        <b>图例</b><br>
        <i class="fa fa-map-marker" style="color:green"></i> 站点1 (PM小)<br>
        <i class="fa fa-map-marker" style="color:red"></i> 站点2 (PM大)<br>
        <span style="color:blue">━━━</span> OSRM 1→2 路径<br>
        <span style="color:red">━━━</span> OSRM 2→1 路径<br>
        <br>
        <b>方向判断不一致:</b><br>
        PM判断与OSRM判断的<br>
        顺向方向相反
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # 保存
    m.save(output_path)
    print(f"地图已保存: {output_path}")
    
    return m


# 生成错误案例地图
if len(mismatch) > 0:
    fwy_suffix = f"_{TARGET_FWY}" if TARGET_FWY else ""
    error_map = create_error_visualization_map(
        mismatch, 
        os.path.join(OUTPUT_DIR, f'direction_mismatch_map{fwy_suffix}.html'),
        max_cases=100
    )
    print(f"\n在浏览器中打开查看地图")
else:
    print("没有方向判断错误的案例")

In [ ]:
# 也生成所有站点对的地图（便于全局查看）
def create_all_pairs_map(valid_pairs_df, output_path, max_pairs=200):
    """
    在地图上可视化所有站点对
    绿色路径: 方向一致
    红色路径: 方向不一致
    灰色路径: OSRM 无法判断方向
    """
    if len(valid_pairs_df) > max_pairs:
        print(f"站点对过多，随机采样 {max_pairs} 个显示")
        valid_pairs_df = valid_pairs_df.sample(n=max_pairs, random_state=42)
    
    center_lat = valid_pairs_df[['Lat1', 'Lat2']].values.mean()
    center_lon = valid_pairs_df[['Lon1', 'Lon2']].values.mean()
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=10,
        tiles='OpenStreetMap'
    )
    
    for idx, row in valid_pairs_df.iterrows():
        # 根据一致性选择颜色
        if row['Dir_by_OSRM'] == 'similar':
            color = 'gray'
        elif row['Direction_Match']:
            color = 'green'
        else:
            color = 'red'
        
        # 只画较短的那条路径（顺向）
        if row['OSRM_1to2_mi'] <= row['OSRM_2to1_mi']:
            geom_str = row['Geometry_1to2']
            dist = row['OSRM_1to2_mi']
        else:
            geom_str = row['Geometry_2to1']
            dist = row['OSRM_2to1_mi']
        
        if geom_str:
            geom = json.loads(geom_str)
            coords = [[c[1], c[0]] for c in geom['coordinates']]
            folium.PolyLine(
                coords,
                color=color,
                weight=3,
                opacity=0.7,
                tooltip=f"{row['ID1']}→{row['ID2']}: {dist:.2f}mi ({row['Type1']}-{row['Type2']})"
            ).add_to(m)
    
    # 图例
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; 
                background-color: white; padding: 10px; border: 2px solid gray;">
        <b>路径颜色</b><br>
        <span style="color:green">━━━</span> 方向一致<br>
        <span style="color:red">━━━</span> 方向不一致<br>
        <span style="color:gray">━━━</span> OSRM无法判断
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))
    
    m.save(output_path)
    print(f"全局地图已保存: {output_path}")
    return m


# 生成全局地图
fwy_suffix = f"_{TARGET_FWY}" if TARGET_FWY else ""
all_map = create_all_pairs_map(
    valid_pairs,
    os.path.join(OUTPUT_DIR, f'all_pairs_map{fwy_suffix}.html'),
    max_pairs=500
)

In [ ]:
# 按类型组合分析错误
if len(mismatch) > 0:
    print("\n按站点类型组合分析错误:")
    print("="*60)
    type_error = mismatch.groupby(['Type1', 'Type2']).size().sort_values(ascending=False)
    print(type_error)
    
    print("\n按高速公路分析错误:")
    print("="*60)
    fwy_error = mismatch.groupby(['Fwy', 'Dir']).size().sort_values(ascending=False)
    print(fwy_error)

In [ ]:
# 单独查看每个错误案例的详细信息
def analyze_single_error(row):
    """分析单个错误案例"""
    print("="*60)
    print(f"错误案例: {row['ID1']} ↔ {row['ID2']}")
    print("="*60)
    
    print(f"\n【高速信息】")
    print(f"  Fwy: {row['Fwy']}-{row['Dir']}")
    
    print(f"\n【站点1】 (PM较小)")
    print(f"  ID: {row['ID1']}")
    print(f"  Type: {row['Type1']}")
    print(f"  Abs_PM: {row['PM1']:.3f}")
    print(f"  坐标: ({row['Lat1']:.5f}, {row['Lon1']:.5f})")
    print(f"  Name: {row['Name1']}")
    
    print(f"\n【站点2】 (PM较大)")
    print(f"  ID: {row['ID2']}")
    print(f"  Type: {row['Type2']}")
    print(f"  Abs_PM: {row['PM2']:.3f}")
    print(f"  坐标: ({row['Lat2']:.5f}, {row['Lon2']:.5f})")
    print(f"  Name: {row['Name2']}")
    
    print(f"\n【距离比较】")
    print(f"  PM 差值: {row['Dist_PM']:.3f} mi")
    print(f"  直线距离: {row['Dist_Geo']:.3f} mi")
    print(f"  OSRM 1→2: {row['OSRM_1to2_mi']:.3f} mi")
    print(f"  OSRM 2→1: {row['OSRM_2to1_mi']:.3f} mi")
    print(f"  OSRM 比值 (大/小): {row['OSRM_Ratio']:.2f}")
    
    print(f"\n【方向判断】")
    print(f"  PM判断: {row['Dir_by_PM']} (基于 Dir={row['Dir']})")
    print(f"  OSRM判断: {row['Dir_by_OSRM']} (基于距离差异)")
    
    print(f"\n【可能原因】")
    if row['Type1'] in ['FR', 'OR'] or row['Type2'] in ['FR', 'OR']:
        print("  → 涉及匝道 (FR/OR)，可能是单向路导致")
    if row['Type1'] == 'HV' or row['Type2'] == 'HV':
        print("  → 涉及 HOV 车道，可能与主线物理隔离")
    if row['OSRM_Ratio'] < 1.5:
        print("  → OSRM 双向距离很接近，方向判断不明确")
    if abs(row['Dist_PM']) < 0.3:
        print("  → PM 差值很小，可能是同位置不同类型站点")


# 分析前几个错误案例
if len(mismatch) > 0:
    print("\n" + "#"*60)
    print("# 错误案例详细分析")
    print("#"*60)
    
    for i in range(min(10, len(mismatch))):
        analyze_single_error(mismatch.iloc[i])
        print("\n")

## 7. 统计分析与可视化

In [ ]:
# 可视化统计
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. OSRM 1to2 vs 2to1
ax = axes[0, 0]
colors = valid_pairs['Direction_Match'].map({True: 'green', False: 'red'})
colors[valid_pairs['Dir_by_OSRM'] == 'similar'] = 'gray'
ax.scatter(valid_pairs['OSRM_1to2_mi'], valid_pairs['OSRM_2to1_mi'], 
           c=colors, alpha=0.5, s=20)
max_val = max(valid_pairs['OSRM_1to2_mi'].max(), valid_pairs['OSRM_2to1_mi'].max())
ax.plot([0, max_val], [0, max_val], 'k--', label='y=x')
ax.set_xlabel('OSRM 1→2 (mi)')
ax.set_ylabel('OSRM 2→1 (mi)')
ax.set_title('OSRM 双向距离\n(绿=一致, 红=不一致, 灰=无法判断)')
ax.grid(True, alpha=0.3)

# 2. OSRM顺向 vs Abs_PM
ax = axes[0, 1]
ax.scatter(valid_pairs['Abs_Dist_PM'], valid_pairs['OSRM_Forward_mi'],
           c=colors, alpha=0.5, s=20)
max_val = max(valid_pairs['Abs_Dist_PM'].max(), valid_pairs['OSRM_Forward_mi'].max())
ax.plot([0, max_val], [0, max_val], 'k--', label='y=x')
ax.set_xlabel('|Abs_PM| (mi)')
ax.set_ylabel('OSRM Forward (mi)')
ax.set_title('OSRM顺向 vs PM距离')
ax.grid(True, alpha=0.3)

# 3. OSRM Ratio 分布
ax = axes[0, 2]
if len(decisive) > 0:
    match_ratio = decisive[decisive['Direction_Match']]['OSRM_Ratio']
    mismatch_ratio = decisive[~decisive['Direction_Match']]['OSRM_Ratio']
    if len(match_ratio) > 0:
        ax.hist(match_ratio, bins=30, alpha=0.7, label=f'一致 (n={len(match_ratio)})', color='green')
    if len(mismatch_ratio) > 0:
        ax.hist(mismatch_ratio, bins=30, alpha=0.7, label=f'不一致 (n={len(mismatch_ratio)})', color='red')
ax.set_xlabel('OSRM Backward/Forward Ratio')
ax.set_ylabel('Count')
ax.set_title('OSRM 双向比值分布')
ax.legend()
ax.set_xlim(1, 10)

# 4. 按距离分组的错误率
ax = axes[1, 0]
if len(decisive) > 0:
    decisive_copy = decisive.copy()
    decisive_copy['Dist_Bin'] = pd.cut(decisive_copy['Dist_Geo'], bins=[0, 0.5, 1, 2, 4])
    error_by_dist = decisive_copy.groupby('Dist_Bin')['Direction_Match'].agg(['sum', 'count'])
    error_by_dist['error_rate'] = (1 - error_by_dist['sum'] / error_by_dist['count']) * 100
    error_by_dist['error_rate'].plot(kind='bar', ax=ax, color='coral')
ax.set_xlabel('距离范围 (mi)')
ax.set_ylabel('错误率 (%)')
ax.set_title('按距离分组的方向判断错误率')
ax.tick_params(axis='x', rotation=45)

# 5. 按站点类型的错误率
ax = axes[1, 1]
if len(decisive) > 0:
    type_stats = decisive.groupby('Type1')['Direction_Match'].agg(['sum', 'count'])
    type_stats['error_rate'] = (1 - type_stats['sum'] / type_stats['count']) * 100
    type_stats['error_rate'].plot(kind='bar', ax=ax, color='steelblue')
ax.set_xlabel('站点1类型')
ax.set_ylabel('错误率 (%)')
ax.set_title('按站点类型的错误率')
ax.tick_params(axis='x', rotation=0)

# 6. 按方向的错误率
ax = axes[1, 2]
if len(decisive) > 0:
    dir_stats = decisive.groupby('Dir')['Direction_Match'].agg(['sum', 'count'])
    dir_stats['error_rate'] = (1 - dir_stats['sum'] / dir_stats['count']) * 100
    dir_stats['error_rate'].plot(kind='bar', ax=ax, color='purple')
ax.set_xlabel('方向')
ax.set_ylabel('错误率 (%)')
ax.set_title('按行驶方向的错误率')
ax.tick_params(axis='x', rotation=0)

fwy_title = f" (Fwy {TARGET_FWY})" if TARGET_FWY else ""
plt.suptitle(f'OSRM 方向判断分析{fwy_title}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'direction_analysis{fwy_suffix}.png'), dpi=150)
plt.show()

## 8. 保存结果

In [ ]:
# 保存完整结果（不含geometry，太大）
save_cols = [
    'ID1', 'ID2', 'Fwy', 'Dir', 'Type1', 'Type2',
    'PM1', 'PM2', 'Dist_PM', 'Dist_Geo',
    'Lat1', 'Lon1', 'Lat2', 'Lon2',
    'OSRM_1to2_mi', 'OSRM_2to1_mi', 'OSRM_Forward_mi', 'OSRM_Backward_mi', 'OSRM_Ratio',
    'Dir_by_PM', 'Dir_by_OSRM', 'Direction_Match',
    'Name1', 'Name2', 'Status_1to2', 'Status_2to1'
]

fwy_suffix = f"_{TARGET_FWY}" if TARGET_FWY else ""

output_file = os.path.join(OUTPUT_DIR, f'osrm_validation{fwy_suffix}.csv')
valid_pairs[save_cols].to_csv(output_file, index=False)
print(f"完整结果已保存: {output_file}")

# 单独保存错误案例
if len(mismatch) > 0:
    error_file = os.path.join(OUTPUT_DIR, f'direction_mismatch{fwy_suffix}.csv')
    mismatch[save_cols].to_csv(error_file, index=False)
    print(f"错误案例已保存: {error_file}")

In [ ]:
# 生成汇总报告
n_decisive = len(decisive) if 'decisive' in dir() and len(decisive) > 0 else 0
n_match = decisive['Direction_Match'].sum() if n_decisive > 0 else 0
n_mismatch = len(mismatch) if 'mismatch' in dir() else 0
n_similar = len(similar) if 'similar' in dir() else 0

report = f"""
OSRM vs Abs_PM 方向判断验证报告
{'='*60}

配置:
  OSRM 服务器: {OSRM_SERVER}
  距离范围: {MIN_GEO_DISTANCE} ~ {MAX_GEO_DISTANCE} mi
  目标高速: {TARGET_FWY if TARGET_FWY else '全部'}
  目标方向: {TARGET_DIR if TARGET_DIR else '全部'}
  元数据文件: {meta_file}

数据概况:
  筛选后站点数: {len(meta_valid)}
  符合条件的站点对: {len(pairs_df)}
  OSRM 双向成功: {len(valid_pairs)}

方向判断结果:
  OSRM有明确方向: {n_decisive}
  方向一致: {n_match} ({n_match/n_decisive*100:.1f}% if n_decisive > 0 else 0)
  方向不一致: {n_mismatch} ({n_mismatch/n_decisive*100:.1f}% if n_decisive > 0 else 0)
  OSRM双向相近: {n_similar}

输出文件:
  - osrm_validation{fwy_suffix}.csv: 完整结果
  - direction_mismatch{fwy_suffix}.csv: 错误案例
  - direction_mismatch_map{fwy_suffix}.html: 错误案例地图 (OSM底图)
  - all_pairs_map{fwy_suffix}.html: 全部路径地图
  - direction_analysis{fwy_suffix}.png: 统计图表
"""

report_file = os.path.join(OUTPUT_DIR, f'osrm_validation_report{fwy_suffix}.txt')
with open(report_file, 'w') as f:
    f.write(report)
print(f"报告已保存: {report_file}")
print(report)

## 9. 总结

### 查看地图

1. **错误案例地图** (`direction_mismatch_map_XX.html`)
   - 只显示方向判断不一致的案例
   - 点击标记查看详细元数据
   - 蓝色=1→2路径，红色=2→1路径

2. **全局地图** (`all_pairs_map_XX.html`)
   - 显示所有站点对
   - 绿色=一致，红色=不一致，灰色=无法判断

### 常见错误原因

1. **匝道单向性**: FR/OR 是单向的
2. **HOV 隔离**: HOV 与主线物理隔离
3. **复杂互通**: 互通处路网复杂
4. **坐标偏差**: 检测器坐标落在错误道路上
5. **双向距离接近**: OSRM 双向距离很接近时判断不可靠